<a href="https://colab.research.google.com/github/svirfneblyn2/genAIcource/blob/main/notebooks/L06_Speech_and_Audio_AI_Pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 06: Speech & Audio AI — STT (Whisper), TTS, Foley SFX & Multi-Stem Mixing
**Course:** GenAI Basics | **Instructor:** Igor Rubanovich (EPAM Systems) | **Models:** Whisper, ElevenLabs, Cartesia, Silero VAD

This notebook provides the complete engineering pipeline for generative audio and speech: from transcribing speech with millisecond word timestamps to synthesizing emotive narration, generating physical Foley sound effects, and programmatically assembling multi-stem soundtracks with Sidechain Ducking.

| Step | Focus | Engineering Concept |
|---|---|---|
| **0** | Environment Setup | Audio libraries (`pydub`, `soundfile`, `whisper`), LUFS targets |
| **1** | Whisper STT & Timestamps | Subtitle generation with word-level Cross-Attention alignment |
| **2** | Silero VAD Preprocessing | Cutting silence intervals to eliminate autoregressive repetition loops |
| **3** | Text-to-Speech (TTS) | Expressive voice synthesis with prosody and breath pacing |
| **4** | Foley SFX Generation | Physics-grounded sound effects synthesis for video impacts |
| **5** | Multi-Stem Assembly | Automated 3-stem mixing with Sidechain Ducking in Python |
| **6** | Audio QA Audit | Automated verification of clipping peaks, LUFS balance, and length |
| **7** | Homework Package Builder | Generating submission files and packaging `genai-homeworks/L06/` |


---
## Step 0: Environment Setup
Install audio manipulation and speech processing libraries.

In [ ]:
!pip install -q pydub soundfile requests numpy
import os, sys, time, json, pathlib
import numpy as np
from pathlib import Path

OUT_DIR = Path('audio_out')
OUT_DIR.mkdir(exist_ok=True)
print(f'[SETUP] Working directory ready: {OUT_DIR.resolve()}')

---
## Step 1: Speech-to-Text (STT) with Word-Level Timestamps
Whisper processes audio in 30-second windows. In `word_timestamps=True` mode, cross-attention weights map exact millisecond start and end boundaries to each word.

In [ ]:
def simulate_whisper_word_timestamps(text, duration_sec=5.0):
    '''Demonstrates word-level timestamp data structure output by Whisper.'''
    words = text.split()
    time_per_word = duration_sec / max(len(words), 1)
    word_list = []
    current_time = 0.0
    for w in words:
        w_start = round(current_time, 2)
        w_end = round(current_time + time_per_word * 0.9, 2)
        word_list.append({'word': w, 'start': w_start, 'end': w_end})
        current_time += time_per_word
    return {'language': 'en', 'duration': duration_sec, 'segments': [{'words': word_list}]}

sample_script = 'The evening rain cuts across the asphalt but order 42 is already en route'
res = simulate_whisper_word_timestamps(sample_script, 4.5)
print(f'Transcription complete ({len(res["segments"][0]["words"])} words):')
for w in res['segments'][0]['words'][:6]:
    print(f'  [{w["start"]:4.2f}s - {w["end"]:4.2f}s]: {w["word"]}')

---
## Step 2: Voice Activity Detection (VAD) Preprocessing
Feeding long intervals of pure silence or room tone into autoregressive Whisper causes hallucination loops (repeating dataset artifacts). Silero VAD strips non-speech regions in 2ms.

In [ ]:
def vad_segment_filter(audio_energy_array, threshold=0.15):
    '''Simulates Silero VAD speech detection over audio frames.'''
    speech_frames = [i for i, energy in enumerate(audio_energy_array) if energy > threshold]
    return f'Detected {len(speech_frames)} active speech frames out of {len(audio_energy_array)}. Silence pruned: {len(audio_energy_array) - len(speech_frames)} frames.'

mock_energy = [0.02, 0.01, 0.45, 0.82, 0.74, 0.65, 0.03, 0.01, 0.02]
print(vad_segment_filter(mock_energy))

---
## Step 3: Text-to-Speech (TTS) Synthesis with Pacing
We formulate an expressive narrator script punctuated for breath and intonation.

In [ ]:
narration_text = 'The evening rain cuts across the asphalt... but order number 42 is already en route... twelve minutes to doorstep.'
print('[SCRIPT READY FOR SYNTHESIS]:')
print(narration_text)
print('\n[ENGINEERING PARAMETERS]: Stability: 0.40 | Similarity Boost: 0.80 | Style Exaggeration: 0.15')

---
## Step 4: Multi-Stem Assembly & Sidechain Ducking in Python
We simulate synthesizing 3 stems (Voiceover, Foley SFX, Music) and applying Sidechain Ducking programmatically.

In [ ]:
import math, wave, struct
from pydub import AudioSegment

def generate_synthetic_stem(filename, freq, duration_s, volume=0.5):
    sample_rate = 44100
    num_samples = int(duration_s * sample_rate)
    with wave.open(str(filename), 'w') as f:
        f.setnchannels(1)
        f.setsampwidth(2)
        f.setframerate(sample_rate)
        for i in range(num_samples):
            val = int(volume * 32767.0 * math.sin(2.0 * math.pi * freq * (i / sample_rate)))
            f.writeframesraw(struct.pack('<h', val))

v_path = OUT_DIR / '01_voiceover.wav'
f_path = OUT_DIR / '02_foley_sfx.wav'
m_path = OUT_DIR / '03_music_bg.wav'

generate_synthetic_stem(v_path, 220.0, 3.5, volume=0.7)
generate_synthetic_stem(f_path, 440.0, 1.5, volume=0.3)
generate_synthetic_stem(m_path, 110.0, 5.0, volume=0.2)

voice = AudioSegment.from_file(v_path)
foley = AudioSegment.from_file(f_path)
music = AudioSegment.from_file(m_path)

# Apply gain balancing
voice = voice.apply_gain(+1.0)
foley = foley.apply_gain(-4.0)

# Sidechain ducking simulation (-5dB on music during voice)
music_ducked = music[:300] + (music[300:300+len(voice)] - 5.0) + music[300+len(voice):]
final_mix = music_ducked.overlay(voice, position=300).overlay(foley, position=1500)

master_out = OUT_DIR / 'final_master.wav'
final_mix.export(master_out, format='wav')
print(f'[SUCCESS] Master audio exported: {master_out} ({master_out.stat().st_size} bytes)')

---
## Step 5: Audio QA Audit
We verify that maximum peak levels never exceed -1.0 dBFS (preventing digital clipping) and verify length consistency.

In [ ]:
mix_segment = AudioSegment.from_file(master_out)
max_peak_dbfs = mix_segment.max_dBFS
duration_sec = len(mix_segment) / 1000.0

print(f'Audio Duration: {duration_sec:.2f} seconds')
print(f'Peak Level:     {max_peak_dbfs:.2f} dBFS')
if max_peak_dbfs <= -1.0:
    print('[QA VERDICT: PASS] Zero digital clipping detected. Headroom preserved.')
else:
    print('[QA VERDICT: WARNING] Peak exceeds -1.0 dBFS limit. Apply limiter.')

---
## Step 6: Packaging Submission for GitHub `genai-homeworks/L06/`

In [ ]:
spec_content = f'''# Audio Design Specification • Homework L06
**Generated by:** Automated Audio Pipeline
**Duration:** {duration_sec:.2f}s
**Peak Headroom:** {max_peak_dbfs:.2f} dBFS
'''
(OUT_DIR / 'audio_spec.md').write_text(spec_content, encoding='utf-8')
print(f'[DONE] Ready for submission in genai-homeworks/L06/')